In [1]:
import os
import glob
import requests
import chromadb
from chromadb.utils import embedding_functions

In [2]:
CLASS_FILES_DIR = "class_files"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "gemma3:12b"
OLLAMA_BASE_URL = "http://localhost:11434"  # Default Ollama URL
EMBEDDING_DIMENSION = 768  # Dimension of nomic-embed-text embeddings (adjust if needed)
CHROMA_PERSIST_DIR = "chroma_db"
COLLECTION_NAME = "class_files_collection"
N_RESULTS = 10

In [3]:
def read_rst_file(filepath):
    """Reads an RST file and returns its content."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return None

def get_embedding(text):
    """Gets the embedding for the given text using Ollama."""
    url = f"{OLLAMA_BASE_URL}/api/embeddings"
    data = {
        "model": EMBEDDING_MODEL,
        "prompt": text
    }
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()
        result = response.json()
        return result['embedding']
    except requests.exceptions.RequestException as e:
        print(f"Error getting embedding from Ollama: {e}")
        return None

def query_llm(prompt):
    """Queries the LLM using Ollama."""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    data = {
        "model": LLM_MODEL,
        "prompt": prompt,
        "stream": False  # Set to True for streaming output
    }
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()
        result = response.json()
        return result['response']
    except requests.exceptions.RequestException as e:
        print(f"Error querying LLM from Ollama: {e}")
        return None

def chunk_text(text, chunk_size=1000, chunk_overlap=100):
    """Simple text chunking function."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - chunk_overlap
    return chunks

def create_chroma_client(persist_directory=CHROMA_PERSIST_DIR):
    """Creates and returns a ChromaDB client."""
    return chromadb.PersistentClient(path=persist_directory)

def get_chroma_collection(client, collection_name=COLLECTION_NAME):
    """Gets or creates a ChromaDB collection."""
    return client.get_or_create_collection(name=collection_name)

In [4]:
# Create ChromaDB client and collection
client = create_chroma_client()
collection = get_chroma_collection(client)

# Index the RST files
if not os.path.exists(CLASS_FILES_DIR):
    print(f"Error: Directory '{CLASS_FILES_DIR}' not found. Please create it and add your RST files.")
else:
    rst_files = glob.glob(os.path.join(CLASS_FILES_DIR, "*.rst"))
    if not rst_files:
        print(f"No RST files found in '{CLASS_FILES_DIR}'.")
    else:
        print("Indexing RST files...")
        for filepath in rst_files:
            filename = os.path.basename(filepath)
            content = read_rst_file(filepath)
            if content:
                chunks = chunk_text(content)
                for i, chunk in enumerate(chunks):
                    embedding = get_embedding(chunk)
                    if embedding:
                        doc_id = f"{filename}_chunk_{i}"
                        collection.add(
                            ids=[doc_id],
                            embeddings=[embedding],
                            documents=[chunk],
                            metadatas={"source": filename, "chunk": i}
                        )
        print("Indexing complete.")

Indexing RST files...


Add of existing embedding ID: following_lines_proportional.rst_chunk_0
Insert of existing embedding ID: following_lines_proportional.rst_chunk_0
Add of existing embedding ID: following_lines_proportional.rst_chunk_1
Insert of existing embedding ID: following_lines_proportional.rst_chunk_1
Add of existing embedding ID: following_lines_proportional.rst_chunk_2
Insert of existing embedding ID: following_lines_proportional.rst_chunk_2
Add of existing embedding ID: following_lines_proportional.rst_chunk_3
Insert of existing embedding ID: following_lines_proportional.rst_chunk_3
Add of existing embedding ID: following_lines_proportional.rst_chunk_4
Insert of existing embedding ID: following_lines_proportional.rst_chunk_4
Add of existing embedding ID: following_lines_proportional.rst_chunk_5
Insert of existing embedding ID: following_lines_proportional.rst_chunk_5
Add of existing embedding ID: following_lines_proportional.rst_chunk_6
Insert of existing embedding ID: following_lines_proportion

Indexing complete.


In [5]:
query = input("Ask a question about the class files: ")

# Get embedding for the query
query_embedding = get_embedding(query)
retrieved_chunks = None

if query_embedding:
    # Search ChromaDB for relevant documents
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=N_RESULTS
    )

    if results and results['documents'] and results['documents'][0]:
        retrieved_chunks = results['documents'][0]
        print("\nRetrieved Chunks:")
        for i, chunk in enumerate(retrieved_chunks):
            print(f"--- Chunk {i+1} ---")
            print(chunk)
        context = "\n\n".join(retrieved_chunks)
    else:
        print("No relevant documents found for your query.")
else:
    print("Could not generate embedding for your query.")


Retrieved Chunks:
--- Chunk 1 ---
ndary by detecting the line, backing up, and turning at a random angle.

Basketball Drills
-----------------

* Understand the concept of classes in Python and how they help organize code.
* Learn about the different parts of a class: attributes, methods, and the constructor (`__init__`).
* Implement a `LineSensor` class to encapsulate the functionality of reading and interpreting data from the robot's reflectance sensors.
* Implement a `LineTracker` class that utilizes the `LineSensor` to perform line-following actions, such as driving until a line is detected.
* Understand how to use lists in Python to store sequences of data.
* Program the XRP robot to perform a "pacer" basketball drill by driving to specified distances and returning to a line.

Following Lines: On-Off Control
-------------------------------

* Understand the limitations of odometry (estimating position based on wheel turns) for precise navigation.
* Learn how reflectance sensors c

In [6]:
if retrieved_chunks:
    # Formulate the prompt for the LLM
    prompt = f"""You are a helpful assistant. Use the following context from class files to answer the user's question. If you don't know the answer, just say you don't know.

    Context:
    {context}

    Question: {query}"""

    # Query the LLM
    print("\nGenerating answer...")
    answer = query_llm(prompt)
    if answer:
        print("\nAnswer:")
        print(answer)
    else:
        print("Could not get an answer from the LLM.")


Generating answer...

Answer:
Okay, let's break down the code for line following, incorporating proportional control as requested. I'm going to provide a complete, runnable example that includes the `LineSensor` and `LineTracker` classes.  I'll add explanations along the way to clarify what's happening.

```python
from XRPLib.defaults import *  # Assumes XRPLib is set up correctly
import time

class LineSensor:
    def __init__(self):
        """Initializes the line sensor by setting up the reflectance sensors."""
        # Replace with actual sensor setup if needed
        self.left_sensor_pin = 1 # Example pin number
        self.right_sensor_pin = 3 # Example pin number
        self.sensor_threshold = 500  # Example threshold

    def left_sensor(self):
        """Reads the value from the left reflectance sensor."""
        # Replace with actual sensor reading code
        # Example:  return digital_read(self.left_sensor_pin)
        return 0  # Dummy value

    def right_sensor(se

In [7]:
if retrieved_chunks:
    # Formulate the prompt for the LLM
    prompt = f"""You are a helpful assistant. Use the following context from class files to answer the user's question. If you don't know the answer, just say you don't know.

    Context:
    {context}

    Question: {query}"""

    # Query the LLM
    print("\nGenerating answer...")
    url = f"{OLLAMA_BASE_URL}/api/generate"
    data = {
        "model": "gemma3:4b",
        "prompt": prompt,
        "stream": False  # Set to True for streaming output
    }
    response = requests.post(url, json=data)
    response.raise_for_status()
    result = response.json()
    answer = result['response']
    if answer:
        print("\nAnswer:")
        print(answer)
    else:
        print("Could not get an answer from the LLM.")


Generating answer...

Answer:
```python
class LineSensor:
    def __init__(self):
        """Initializes the line sensor by setting up the reflectance sensors."""
        # Simulate reflectance sensor readings (replace with actual sensor readings)
        self.left_sensor_value = 0
        self.right_sensor_value = 0

    def left_sensor(self):
        # Simulate left sensor reading (replace with actual sensor reading)
        return self.left_sensor_value

    def right_sensor(self):
        # Simulate right sensor reading (replace with actual sensor reading)
        return self.right_sensor_value

class LineTracker:
    def __init__(self, drivetrain, kp=0.1):
        """Initializes the line tracker with a drivetrain and proportional gain."""
        self.drivetrain = drivetrain
        self.kp = kp  # Proportional gain
        self.sensor = LineSensor()  # LineSensor object

    def get_error(self):
        """Calculates the error between the left and right sensor readings."""
     

In [8]:
if retrieved_chunks:
    # Formulate the prompt for the LLM
    prompt = f"""You are a helpful assistant. Use the following context from class files to answer the user's question. If you don't know the answer, just say you don't know.

    Context:
    {context}

    Question: {query}"""

    # Query the LLM
    print("\nGenerating answer...")
    url = f"{OLLAMA_BASE_URL}/api/generate"
    data = {
        "model": "gemma3:1b",
        "prompt": prompt,
        "stream": False  # Set to True for streaming output
    }
    response = requests.post(url, json=data)
    response.raise_for_status()
    result = response.json()
    answer = result['response']
    if answer:
        print("\nAnswer:")
        print(answer)
    else:
        print("Could not get an answer from the LLM.")


Generating answer...

Answer:
```python
from XRPLib.defaults import *

class LineSensor:
    def __init__(self):
        self.left_sensor = re

    def turn_degrees(self, degrees):
        print(f"Turning {degrees} degrees")
        return degrees

class LineTracker:
    def __init__(self, drivetrain):
        self.sensor = LineSensor()
        self.drivetrain = drivetrain

    def proportional_signal(self, KP, base_speed):
        """Generates motor speeds using proportional control based on the error between the sen

    print("LineTracker initialized")
    
    # Example Usage
    
    # Simulate sensor readings
    
    # set KP to 0.8 and base speed to 10
    
    #  print("KP: 0.8, base speed: 10")
    
    
    print("LineTracker is initialized")
    
    # Example:
    
    # line_tracker.proportional_signal(0.8, 10)
    
    # Print the calculated motor speeds
    
    print("LineTracker is now running")
    
    
    print("End of Example")
    
    
    print("To implement 